<a href="https://colab.research.google.com/github/Ed-Neema/movie_sentiment_classifier/blob/main/text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Before you begin, make sure you have all the necessary libraries installed:

In [ ]:
pip install --upgrade transformers datasets evaluate accelerate

In [ ]:
#logging in to HF
from huggingface_hub import notebook_login
notebook_login()

### Load IMDb dataset
Start by loading the IMDb dataset from the 🤗 Datasets library:

In [ ]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Examine the data

There are two fields in this dataset:

*   text: the movie review text.
*   label: a value that is either 0 for a negative review or 1 for a positive review.






In [ ]:
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
imdb["test"][0]

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

## Preprocessing
The next step is to load a DistilBERT tokenizer to preprocess the text field:

- A **tokenizer** is the tool that chops text into pieces (tokens) and converts each piece into a number (an ID)

- *You must use the exact tokenizer that matches your model.*

- DistilBERT learned English using one specific vocabulary — a fixed dictionary where every known word-piece has an assigned ID number. If you fed it text numbered by a different scheme, the numbers would mean nothing to it, like handing someone a book written in an alphabet they've never seen.
- `from_pretrained("distilbert/distilbert-base-uncased")` downloads that model's own matching vocabulary so the numbers line up with what it learned.
-  the tokenizer turns "This movie was great" into something like [101, 2023, 3185, 2001, 2307, 102] — each number is a token's ID. (The 101 and 102 are special markers for "start" and "end.")






In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Create a preprocessing function to tokenize text and truncate sequences to be no longer than DistilBERT’s maximum input length:


The code below just wraps the tokenizer in a reusable function so you can apply it to the whole dataset. Two things to notice:

- `examples["text"]` — it pulls the `text` field out of each review (ignoring the `label` field, which is already a number and doesn't need tokenizing).
- `truncation=True` — DistilBERT has a maximum length it can accept (512 tokens). Some movie reviews are longer than that. `truncation=True` says "if a review is too long, cut off the end so it fits." Without this, an overly long review would crash the model.

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

- `map` runs your function over every single review in the dataset, producing a new version (`tokenized_imdb`) where each review now has its number-lists attached. batched=True is purely a speed trick — instead of tokenizing one review at a time, it hands the tokenizer a chunk of reviews at once, which is much faster. Same result, just quicker.

After this step, every review carries its input_ids (the token numbers) alongside the original text and label.


In [ ]:
tokenized_imdb = imdb.map(preprocess_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Now create a batch of examples using DataCollatorWithPadding. It’s more efficient to dynamically pad the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

This one solves a subtle problem. The model processes reviews in small groups called **batches**, and for the math to work, *every review in the same batch must be the same length.* But reviews are naturally different lengths — one might be 40 tokens, another **300**.

The fix is padding: *adding meaningless filler tokens to the shorter ones so they all match the longest in the batch*. So a batch might get padded to length 300, with the 40-token review getting 260 filler tokens tacked on.

The clever bit is the word ***dynamic*** in the tutorial's note. You could pad every review in the entire dataset to the maximum 512 — but then your 40-token review wastes 472 slots of computation on filler. Instead, the **collator pads per batch, only up to the longest review in that particular batch**. If a batch's longest review is 300 tokens, everything's padded to 300, not 512. Less wasted computation, faster training. The collator isn't run now — it's an instruction that gets used later, during training, to assemble each batch on the fly.



In [ ]:
from transformers import DataCollatorWithPadding
#creating a padding tool and configuring it.
"""
The collator needs the tokenizer for two reasons:
- to know which token ID means "padding" (each model reserves a specific number for filler),
- to know which side to pad on. So DataCollatorWithPadding(tokenizer=tokenizer) means "make me a padding machine set up for DistilBERT's rules."
"""
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Evaluate
 You can quickly load a evaluation method with the 🤗 Evaluate library.

 For this task, load the accuracy metric

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

Then create a function that passes your predictions and labels to compute to calculate the accuracy:

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

#Your compute_metrics function is ready to go now, and you’ll return to it when you setup your training.

## Train
Before you start training your model, create a map of the expected ids to their labels with id2label and label2id:

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

You’re ready to start training your model now! Load DistilBERT with AutoModelForSequenceClassification along with the number of expected labels, and the label mappings:

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
